# STAGE 1 – TÍCH HỢP DỮ LIỆU MỚI: CONCAT & MERGE

Notebook cập nhật quy trình tích hợp CGV với 3 thuộc tính AI bổ sung: `TicketPrice`, `TotalSeats`, `AvailableSeats`.

Quy trình:
1. Đọc dữ liệu phim đang chiếu, phim sắp chiếu và lịch chiếu.
2. Dùng `pd.concat()` nối dọc hai bảng phim.
3. Bổ sung 3 thuộc tính AI vào lịch chiếu bằng 5 khóa suất chiếu nếu chưa có file v2.
4. Dùng `pd.merge()` gộp thông tin phim vào lịch chiếu theo `MovieName`.
5. Kiểm tra và xuất `cgv_final_integrated_v2.csv`.

> TicketPrice, TotalSeats và AvailableSeats là dữ liệu mô phỏng do AI tạo sinh, không phải dữ liệu vận hành thực tế của CGV.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

DATA_DIR = Path(".")
DETAIL_FILE = DATA_DIR / "cgv_movie_details.csv"
UPCOMING_FILE = DATA_DIR / "cgv_upcoming_movies.csv"
SHOWTIMES_FILE = DATA_DIR / "cgv_showtimes_AI_generated.csv"
AI_EXTRA_FILE = DATA_DIR / "AI_tao_sinh_2.csv"
SHOWTIMES_V2_FILE = DATA_DIR / "cgv_showtimes_AI_generated_v2.csv"

STATIC_OUTPUT_FILE = DATA_DIR / "cgv_static_master_v2.csv"
FINAL_OUTPUT_FILE = DATA_DIR / "cgv_final_integrated_v2.csv"

for f in [DETAIL_FILE, UPCOMING_FILE, SHOWTIMES_FILE]:
    print(f"{f.name}: {'OK' if f.exists() else 'KHÔNG TÌM THẤY'}")
if not all(f.exists() for f in [DETAIL_FILE, UPCOMING_FILE, SHOWTIMES_FILE]):
    raise FileNotFoundError("Hãy đặt 3 file CSV đầu vào trong cùng thư mục với notebook.")

cgv_movie_details.csv: OK
cgv_upcoming_movies.csv: OK
cgv_showtimes_AI_generated.csv: OK


## 1. Đọc dữ liệu

In [2]:
df_detail = pd.read_csv(DETAIL_FILE, encoding="utf-8-sig")
df_upcoming = pd.read_csv(UPCOMING_FILE, encoding="utf-8-sig")
df_showtimes = pd.read_csv(SHOWTIMES_FILE, encoding="utf-8-sig")

print("Phim đang chiếu:", df_detail.shape)
print("Phim sắp chiếu:", df_upcoming.shape)
print("Lịch chiếu:", df_showtimes.shape)
display(df_detail.head())
display(df_upcoming.head())
display(df_showtimes.head())

Phim đang chiếu: (30, 9)
Phim sắp chiếu: (34, 9)
Lịch chiếu: (1172, 6)


,MovieName,MovieURL,Rating,Director,Actors,Genre,ReleaseDate,Duration,Language
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),https://www.cgv.vn/default/avengers-endgame-encore.html,T13,"Anthony Russo, Joe Russo","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner...","Hành Động, Khoa Học Viễn Tưởng, Phiêu Lưu",25/09/2026,183 phút,Tiếng Anh với phụ đề tiếng Việt
1,TRẠI BUÔN NGƯỜI,https://www.cgv.vn/default/trai-buon-nguoi.html,T18,Toni Dương Bảo Anh,"Steven Nguyễn, Quách Ngọc Ngoan, Tùng Mint, Huỳnh Minh Kiên,...","Hành Động, Hồi hộp",25/09/2026,135 phút,Tiếng Việt
2,VÙNG ĐẤT QUỶ DỮ,https://www.cgv.vn/default/resident-evil-2026.html,T18,Zach Cregger,"Austin Abrams,Paul Walter Hauser, Zach Cherry, Kali Reis.","Hành Động, Kinh Dị",18/09/2026,94 phút,Tiếng Anh - Phụ đề Tiếng Việt
3,HÒN ĐẢO QUÊN LÃNG,https://www.cgv.vn/default/forgotten-island.html,K,Joel Crawford; Januel Mercado,"H.E.R., Liza Soberano, Dave Franco, Jenny Slate, Manny Jacinto, Dolly de Leon, Jo Koy, Ronny Chi...","Hoạt Hình, Phiêu Lưu",25/09/2026,109 phút,Phụ đề Tiếng Việt & Lồng Tiếng Việt
4,ÚT LAN 2,https://www.cgv.vn/default/ut-lan-2.html,T18,Trần Trọng Dần,"Phương Nam, Kiều Oanh, NSUT Kim Phương, Kiều Diễm, Gia Tuệ, Tạ Lâm, Ngọc Phước, Khánh Tiên, Tấn ...",Kinh Dị,25/09/2026,107 minutes,Vietnamese


,MovieName,MovieURL,Rating,Director,Actors,Genre,ReleaseDate,Duration,Language
0,TRÁI TIM QUÁI THÚ,https://www.cgv.vn/default/heart-of-the-beast.html,T13,David Ayer,"Brad Pitt, J.K. Simmons, Anna Lambe",Hành Động,02/10/2026,101 phút,Tiếng Anh - Phụ đề Tiếng Việt
1,THẦN SƯ CHUNG QUỲ: LINH GIỚI ĐẠI CHIẾN,https://www.cgv.vn/default/master-zhong.html,NaN,Wang Yuxi and Huang Shanchuan,"Duan Yixuan, Yu Haowei, Hu Liangwei, Zhao Chengchen, Ba He, and Hu Zhengjian.","Hành Động, Hoạt Hình, Phiêu Lưu, Thần thoại",02/10/2026,96 phút,Tiếng Trung Quốc - Phụ đề Tiếng Việt
2,KHÓA CHẶT CỬA NÀO SUZUME,https://www.cgv.vn/default/suzume.html,P,Makoto Shinkai,NaN,"Hoạt Hình, Phiêu Lưu",02/10/2026,122 phút,"Tiếng Nhật - Phụ đề Tiếng Việt, tiếng Anh"
3,QUYẾT CUA ANH NÀY,https://www.cgv.vn/default/quyet-cua-anh-nay.html,NaN,Mez Tharatorn,"Nadech Kugimiya, Minnie Nicha, Pa Kiat Kitcharoen, Ble Patumrach, Vivian Thipakorn","Hài, Tình cảm",02/10/2026,NaN,Tiếng Thái - Phụ đề tiếng Việt
4,ÁN MẠNG KARAOKE,https://www.cgv.vn/default/an-mang-karaoke.html,T16,Hoàng Lê,"Đoàn Thiên Ân, Đại Nghĩa, Vân Dung,...","Hài, Hồi hộp",02/10/2026,96 phút,Tiếng Việt


,MovieName,Cinema,ShowDate,Format,ShowTime,GeneratedText
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,10:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...
1,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,16:00,"Hôm nay, tôi có cơ hội tham gia vào một trải nghiệm thú vị khi xem suất chiếu ""AVENGERS: HỒI KẾT..."
2,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,21:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...
3,CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI CÁ,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh & Việt,18:00,"Hôm nay, tôi đã có cơ hội đến rạp chiếu CGV Crescent Mall để xem phim CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI..."
4,HÒN ĐẢO QUÊN LÃNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,19:40,"Tôi vừa có cơ hội xem suất chiếu ""HÒN ĐẢO QUÊN LÃNG"" tại rạp chiếu CGV Crescent Mall và trải ngh..."


## 2. Chuẩn hóa dữ liệu và khóa suất chiếu

In [3]:
def normalize_movie_name(data):
    data = data.copy()
    if "MovieName" not in data.columns:
        raise KeyError("Thiếu cột MovieName")
    data["MovieName"] = data["MovieName"].astype("string").str.strip()
    return data

df_detail = normalize_movie_name(df_detail)
df_upcoming = normalize_movie_name(df_upcoming)
df_showtimes = normalize_movie_name(df_showtimes)

KEY_COLUMNS = ["MovieName", "Cinema", "ShowDate", "Format", "ShowTime"]
missing = [c for c in KEY_COLUMNS if c not in df_showtimes.columns]
if missing:
    raise KeyError(f"Thiếu cột khóa lịch chiếu: {missing}")

for col in KEY_COLUMNS:
    df_showtimes[col] = df_showtimes[col].astype("string").str.strip()

print("Số dòng lịch chiếu:", len(df_showtimes))
print("Số khóa suất chiếu duy nhất:", len(df_showtimes.drop_duplicates(subset=KEY_COLUMNS)))
print("Số khóa trùng:", df_showtimes.duplicated(subset=KEY_COLUMNS).sum())

Số dòng lịch chiếu: 1172
Số khóa suất chiếu duy nhất: 1172
Số khóa trùng: 0


## 3. Bổ sung 3 thuộc tính AI vào lịch chiếu

Ưu tiên sử dụng `cgv_showtimes_AI_generated_v2.csv` nếu đã có. Nếu chưa có, notebook sẽ `merge` `AI_tao_sinh_2.csv` vào lịch chiếu bằng 5 khóa: `MovieName`, `Cinema`, `ShowDate`, `Format`, `ShowTime`.

Đây là merge ngang vì ba thuộc tính mới mô tả các suất chiếu hiện có.

In [4]:
if SHOWTIMES_V2_FILE.exists():
    print("Sử dụng dataset mới:", SHOWTIMES_V2_FILE.name)
    df_showtimes_new = pd.read_csv(SHOWTIMES_V2_FILE, encoding="utf-8-sig")
else:
    if not AI_EXTRA_FILE.exists():
        raise FileNotFoundError("Cần có cgv_showtimes_AI_generated_v2.csv hoặc AI_tao_sinh_2.csv. Hãy chạy AI_tao_sinh_2.ipynb trước.")
    df_ai = pd.read_csv(AI_EXTRA_FILE, encoding="utf-8-sig")
    AI_COLUMNS = KEY_COLUMNS + ["TicketPrice", "AvailableSeats", "TotalSeats"]
    missing_ai = [c for c in AI_COLUMNS if c not in df_ai.columns]
    if missing_ai:
        raise KeyError(f"Bảng AI thiếu cột: {missing_ai}")
    for col in KEY_COLUMNS:
        df_ai[col] = df_ai[col].astype("string").str.strip()
    if df_showtimes.duplicated(subset=KEY_COLUMNS).any():
        raise ValueError("Lịch chiếu có khóa trùng, cần xử lý trước khi merge.")
    if df_ai.duplicated(subset=KEY_COLUMNS).any():
        raise ValueError("Bảng AI có khóa trùng, cần xử lý trước khi merge.")
    df_showtimes_new = df_showtimes.merge(
        df_ai[AI_COLUMNS], on=KEY_COLUMNS, how="left", validate="one_to_one"
    )
    df_showtimes_new.to_csv(SHOWTIMES_V2_FILE, index=False, encoding="utf-8-sig")
    print("Đã tạo:", SHOWTIMES_V2_FILE.name)

print("Kích thước lịch chiếu mới:", df_showtimes_new.shape)
print("Các cột:", df_showtimes_new.columns.tolist())
display(df_showtimes_new.head())

Sử dụng dataset mới: cgv_showtimes_AI_generated_v2.csv
Kích thước lịch chiếu mới: (1172, 9)
Các cột: ['MovieName', 'Cinema', 'ShowDate', 'Format', 'ShowTime', 'GeneratedText', 'TicketPrice', 'AvailableSeats', 'TotalSeats']


,MovieName,Cinema,ShowDate,Format,ShowTime,GeneratedText,TicketPrice,AvailableSeats,TotalSeats
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,10:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...,70000,80,100
1,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,16:00,"Hôm nay, tôi có cơ hội tham gia vào một trải nghiệm thú vị khi xem suất chiếu ""AVENGERS: HỒI KẾT...",100000,90,100
2,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,21:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...,90000,80,100
3,CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI CÁ,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh & Việt,18:00,"Hôm nay, tôi đã có cơ hội đến rạp chiếu CGV Crescent Mall để xem phim CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI...",80000,90,100
4,HÒN ĐẢO QUÊN LÃNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,19:40,"Tôi vừa có cơ hội xem suất chiếu ""HÒN ĐẢO QUÊN LÃNG"" tại rạp chiếu CGV Crescent Mall và trải ngh...",100000,50,80


## 4. Nối dọc hai bảng phim bằng `pd.concat()`

Hai bảng có cùng cấu trúc cột nên có thể nối dọc. `ignore_index=True` giúp đánh lại chỉ số dòng.

In [5]:
if set(df_detail.columns) != set(df_upcoming.columns):
    raise ValueError("Hai bảng phim không cùng bộ cột. Cần chuẩn hóa trước khi concat.")

df_upcoming = df_upcoming[df_detail.columns]
df_static = pd.concat([df_detail, df_upcoming], axis=0, ignore_index=True)

print("Kích thước phim đang chiếu:", df_detail.shape)
print("Kích thước phim sắp chiếu:", df_upcoming.shape)
print("Kích thước sau concat:", df_static.shape)
display(df_static.head())

Kích thước phim đang chiếu: (30, 9)
Kích thước phim sắp chiếu: (34, 9)
Kích thước sau concat: (64, 9)


,MovieName,MovieURL,Rating,Director,Actors,Genre,ReleaseDate,Duration,Language
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),https://www.cgv.vn/default/avengers-endgame-encore.html,T13,"Anthony Russo, Joe Russo","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner...","Hành Động, Khoa Học Viễn Tưởng, Phiêu Lưu",25/09/2026,183 phút,Tiếng Anh với phụ đề tiếng Việt
1,TRẠI BUÔN NGƯỜI,https://www.cgv.vn/default/trai-buon-nguoi.html,T18,Toni Dương Bảo Anh,"Steven Nguyễn, Quách Ngọc Ngoan, Tùng Mint, Huỳnh Minh Kiên,...","Hành Động, Hồi hộp",25/09/2026,135 phút,Tiếng Việt
2,VÙNG ĐẤT QUỶ DỮ,https://www.cgv.vn/default/resident-evil-2026.html,T18,Zach Cregger,"Austin Abrams,Paul Walter Hauser, Zach Cherry, Kali Reis.","Hành Động, Kinh Dị",18/09/2026,94 phút,Tiếng Anh - Phụ đề Tiếng Việt
3,HÒN ĐẢO QUÊN LÃNG,https://www.cgv.vn/default/forgotten-island.html,K,Joel Crawford; Januel Mercado,"H.E.R., Liza Soberano, Dave Franco, Jenny Slate, Manny Jacinto, Dolly de Leon, Jo Koy, Ronny Chi...","Hoạt Hình, Phiêu Lưu",25/09/2026,109 phút,Phụ đề Tiếng Việt & Lồng Tiếng Việt
4,ÚT LAN 2,https://www.cgv.vn/default/ut-lan-2.html,T18,Trần Trọng Dần,"Phương Nam, Kiều Oanh, NSUT Kim Phương, Kiều Diễm, Gia Tuệ, Tạ Lâm, Ngọc Phước, Khánh Tiên, Tấn ...",Kinh Dị,25/09/2026,107 minutes,Vietnamese


## 5. Kiểm tra khóa `MovieName` trong bảng tĩnh

Một phim có thể xuất hiện ở cả hai bảng. Loại bỏ tên phim trùng để đảm bảo bảng bên phải có một dòng cho mỗi `MovieName`, tránh nhân bản suất chiếu khi merge.

In [6]:
duplicate_movies = df_static[df_static.duplicated(subset=["MovieName"], keep=False)]
print("Số dòng có MovieName trùng:", len(duplicate_movies))
if not duplicate_movies.empty:
    display(duplicate_movies.sort_values("MovieName").head(20))

df_static = df_static.drop_duplicates(subset=["MovieName"], keep="first").copy()
print("Số phim duy nhất:", df_static["MovieName"].nunique())
print("Kích thước bảng tĩnh:", df_static.shape)

df_static.to_csv(STATIC_OUTPUT_FILE, index=False, encoding="utf-8-sig")
print("Đã lưu:", STATIC_OUTPUT_FILE.name)

Số dòng có MovieName trùng: 0
Số phim duy nhất: 64
Kích thước bảng tĩnh: (64, 9)
Đã lưu: cgv_static_master_v2.csv


## 6. Gộp ngang bảng tĩnh với lịch chiếu bằng `pd.merge()`

Dùng `MovieName` làm khóa. Chọn `how="left"` để giữ toàn bộ suất chiếu; `validate="many_to_one"` kiểm tra rằng nhiều suất chiếu có thể thuộc cùng một phim, nhưng bảng phim chỉ có một dòng cho mỗi tên phim.

In [7]:
df_final = df_showtimes_new.merge(
    df_static,
    on="MovieName",
    how="left",
    suffixes=("_showtime", "_movie"),
    validate="many_to_one",
    indicator=True
)

print("Trước merge:", df_showtimes_new.shape)
print("Sau merge:", df_final.shape)
print("Kết quả ghép:")
print(df_final["_merge"].value_counts())

df_final = df_final.drop(columns=["_merge"])
display(df_final.head())

Trước merge: (1172, 9)
Sau merge: (1172, 18)
Kết quả ghép:
_merge
both          1172
left_only        0
right_only       0
Name: count, dtype: int64


,MovieName,Cinema,ShowDate,Format,ShowTime,GeneratedText,TicketPrice,AvailableSeats,TotalSeats,MovieURL,Rating,Director,Actors,Genre,ReleaseDate,Duration,Language
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,10:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...,70000,80,100,https://www.cgv.vn/default/avengers-endgame-encore.html,T13,"Anthony Russo, Joe Russo","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner...","Hành Động, Khoa Học Viễn Tưởng, Phiêu Lưu",25/09/2026,183 phút,Tiếng Anh với phụ đề tiếng Việt
1,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,16:00,"Hôm nay, tôi có cơ hội tham gia vào một trải nghiệm thú vị khi xem suất chiếu ""AVENGERS: HỒI KẾT...",100000,90,100,https://www.cgv.vn/default/avengers-endgame-encore.html,T13,"Anthony Russo, Joe Russo","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner...","Hành Động, Khoa Học Viễn Tưởng, Phiêu Lưu",25/09/2026,183 phút,Tiếng Anh với phụ đề tiếng Việt
2,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,21:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...,90000,80,100,https://www.cgv.vn/default/avengers-endgame-encore.html,T13,"Anthony Russo, Joe Russo","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner...","Hành Động, Khoa Học Viễn Tưởng, Phiêu Lưu",25/09/2026,183 phút,Tiếng Anh với phụ đề tiếng Việt
3,CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI CÁ,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh & Việt,18:00,"Hôm nay, tôi đã có cơ hội đến rạp chiếu CGV Crescent Mall để xem phim CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI...",80000,90,100,https://www.cgv.vn/default/chiikawa-the-movie.html,p,Kei Oikawa,"Haruka Aoki, Makoto Tanaka, Ari Ozawa, Yuka Iguchi, Takayuki Asai, Yuma Uchida, Miyuri Shimabuku...",Hoạt Hình,28/08/2026,99 phút,Tiếng Nhật với phụ đề Việt & Anh; Lồng tiếng Việt
4,HÒN ĐẢO QUÊN LÃNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,19:40,"Tôi vừa có cơ hội xem suất chiếu ""HÒN ĐẢO QUÊN LÃNG"" tại rạp chiếu CGV Crescent Mall và trải ngh...",100000,50,80,https://www.cgv.vn/default/forgotten-island.html,K,Joel Crawford; Januel Mercado,"H.E.R., Liza Soberano, Dave Franco, Jenny Slate, Manny Jacinto, Dolly de Leon, Jo Koy, Ronny Chi...","Hoạt Hình, Phiêu Lưu",25/09/2026,109 phút,Phụ đề Tiếng Việt & Lồng Tiếng Việt


## 7. Kiểm tra kết quả tích hợp

In [8]:
print("Số dòng trước merge:", len(df_showtimes_new))
print("Số dòng sau merge:", len(df_final))
print("Số dòng được giữ nguyên:", len(df_final) == len(df_showtimes_new))

for col in ["TicketPrice", "AvailableSeats", "TotalSeats"]:
    if col in df_final.columns:
        print(f"Số giá trị thiếu {col}:", df_final[col].isna().sum())

if "TicketPrice" in df_final.columns:
    print("TicketPrice trong khoảng 75.000–150.000:",
          df_final["TicketPrice"].dropna().between(75000, 150000).all())

if {"AvailableSeats", "TotalSeats"}.issubset(df_final.columns):
    valid = df_final["AvailableSeats"].notna() & df_final["TotalSeats"].notna()
    print("AvailableSeats không âm:",
          (df_final.loc[valid, "AvailableSeats"] >= 0).all())
    print("AvailableSeats <= TotalSeats:",
          (df_final.loc[valid, "AvailableSeats"] <= df_final.loc[valid, "TotalSeats"]).all())

display(df_final.isna().sum().sort_values(ascending=False).to_frame("Missing Values"))

Số dòng trước merge: 1172
Số dòng sau merge: 1172
Số dòng được giữ nguyên: True
Số giá trị thiếu TicketPrice: 0
Số giá trị thiếu AvailableSeats: 0
Số giá trị thiếu TotalSeats: 0
TicketPrice trong khoảng 75.000–150.000: False
AvailableSeats không âm: True
AvailableSeats <= TotalSeats: True


,Missing Values
MovieName,0
MovieURL,0
Duration,0
ReleaseDate,0
Genre,0
Actors,0
Director,0
Rating,0
TotalSeats,0
Cinema,0


## 8. Xuất dataset tích hợp cuối cùng

In [9]:
df_final.to_csv(FINAL_OUTPUT_FILE, index=False, encoding="utf-8-sig")
print("Đã xuất:", FINAL_OUTPUT_FILE.resolve())
print("Kích thước dataset cuối cùng:", df_final.shape)
print("Danh sách cột:")
print(df_final.columns.tolist())
display(df_final.head(10))

Đã xuất: D:\Project_CGV_Group3_backups\STAGE_1\cgv_final_integrated_v2.csv
Kích thước dataset cuối cùng: (1172, 17)
Danh sách cột:
['MovieName', 'Cinema', 'ShowDate', 'Format', 'ShowTime', 'GeneratedText', 'TicketPrice', 'AvailableSeats', 'TotalSeats', 'MovieURL', 'Rating', 'Director', 'Actors', 'Genre', 'ReleaseDate', 'Duration', 'Language']


,MovieName,Cinema,ShowDate,Format,ShowTime,GeneratedText,TicketPrice,AvailableSeats,TotalSeats,MovieURL,Rating,Director,Actors,Genre,ReleaseDate,Duration,Language
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,10:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...,70000,80,100,https://www.cgv.vn/default/avengers-endgame-encore.html,T13,"Anthony Russo, Joe Russo","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner...","Hành Động, Khoa Học Viễn Tưởng, Phiêu Lưu",25/09/2026,183 phút,Tiếng Anh với phụ đề tiếng Việt
1,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,16:00,"Hôm nay, tôi có cơ hội tham gia vào một trải nghiệm thú vị khi xem suất chiếu ""AVENGERS: HỒI KẾT...",100000,90,100,https://www.cgv.vn/default/avengers-endgame-encore.html,T13,"Anthony Russo, Joe Russo","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner...","Hành Động, Khoa Học Viễn Tưởng, Phiêu Lưu",25/09/2026,183 phút,Tiếng Anh với phụ đề tiếng Việt
2,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU LẠI),CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,21:30,Tôi đã có cơ hội xem suất chiếu AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT tại rạp chiếu CGV Crescen...,90000,80,100,https://www.cgv.vn/default/avengers-endgame-encore.html,T13,"Anthony Russo, Joe Russo","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner...","Hành Động, Khoa Học Viễn Tưởng, Phiêu Lưu",25/09/2026,183 phút,Tiếng Anh với phụ đề tiếng Việt
3,CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI CÁ,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh & Việt,18:00,"Hôm nay, tôi đã có cơ hội đến rạp chiếu CGV Crescent Mall để xem phim CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI...",80000,90,100,https://www.cgv.vn/default/chiikawa-the-movie.html,p,Kei Oikawa,"Haruka Aoki, Makoto Tanaka, Ari Ozawa, Yuka Iguchi, Takayuki Asai, Yuma Uchida, Miyuri Shimabuku...",Hoạt Hình,28/08/2026,99 phút,Tiếng Nhật với phụ đề Việt & Anh; Lồng tiếng Việt
4,HÒN ĐẢO QUÊN LÃNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,19:40,"Tôi vừa có cơ hội xem suất chiếu ""HÒN ĐẢO QUÊN LÃNG"" tại rạp chiếu CGV Crescent Mall và trải ngh...",100000,50,80,https://www.cgv.vn/default/forgotten-island.html,K,Joel Crawford; Januel Mercado,"H.E.R., Liza Soberano, Dave Franco, Jenny Slate, Manny Jacinto, Dolly de Leon, Jo Koy, Ronny Chi...","Hoạt Hình, Phiêu Lưu",25/09/2026,109 phút,Phụ đề Tiếng Việt & Lồng Tiếng Việt
5,LAPUTA: LÂU ĐÀI TRÊN KHÔNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,12:50,"Hôm nay, tôi đã có cơ hội xem suất chiếu LAPUTA: LÂU ĐÀI TRÊN KHÔNG tại rạp chiếu CGV Crescent M...",80000,70,100,https://www.cgv.vn/default/laputa.html,K,Hayao Miyazaki,"Mayumi Tanaka, Keiko Yokozawa, Kotoe Hatsui, Minori Terada, Ichiro Nagai,...",Hoạt Hình,25/09/2026,129 phút,Tiếng Nhật - Phụ đề Tiếng Việt; Lồng Tiếng Việt
6,LAPUTA: LÂU ĐÀI TRÊN KHÔNG,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt,14:40,"Hôm nay, tôi đã có cơ hội xem suất chiếu LAPUTA: LÂU ĐÀI TRÊN KHÔNG tại rạp CGV Crescent Mall. C...",90000,80,100,https://www.cgv.vn/default/laputa.html,K,Hayao Miyazaki,"Mayumi Tanaka, Keiko Yokozawa, Kotoe Hatsui, Minori Terada, Ichiro Nagai,...",Hoạt Hình,25/09/2026,129 phút,Tiếng Nhật - Phụ đề Tiếng Việt; Lồng Tiếng Việt
7,LAPUTA: LÂU ĐÀI TRÊN KHÔNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,17:10,Tôi vừa có cơ hội xem suất chiếu LAPUTA: LÂU ĐÀI TRÊN KHÔNG tại rạp chiếu CGV Crescent Mall và đ...,100000,60,80,https://www.cgv.vn/default/laputa.html,K,Hayao Miyazaki,"Mayumi Tanaka, Keiko Yokozawa, Kotoe Hatsui, Minori Terada, Ichiro Nagai,...",Hoạt Hình,25/09/2026,129 phút,Tiếng Nhật - Phụ đề Tiếng Việt; Lồng Tiếng Việt
8,LÊN HƯƠNG,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh,09:35,Tôi đã trải nghiệm một buổi chiếu phim tuyệt vời tại rạp CGV Crescent Mall với bộ phim LÊN HƯƠNG...,90

## 9. Giải thích hai phép tích hợp

- **`concat()`**: nối dọc dữ liệu phim đang chiếu và phim sắp chiếu vì hai bảng có cùng cấu trúc cột.
- **`merge()` lần 1**: thêm `TicketPrice`, `AvailableSeats`, `TotalSeats` vào từng suất chiếu bằng 5 khóa để tránh ghép nhầm các suất của cùng một phim.
- **`merge()` lần 2**: gộp thông tin phim vào lịch chiếu theo `MovieName`.
- **LEFT JOIN**: giữ lại toàn bộ suất chiếu làm dữ liệu trung tâm.

